In [ ]:
# ==============================================================================
# DualBlind AI Benchmark - Colab Multi-Model Serving Node
# Models: mistral-nemo:12b, qwen2.5:14b, codellama:7b, llama3.2:3b
# ==============================================================================
import os, subprocess, time, urllib.request, re

print("1/4 Installing Ollama & system packages...")
subprocess.run("apt-get update -qq && apt-get install -y -qq zstd pciutils curl", shell=True, check=True)
subprocess.run("curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst -o /tmp/ollama.tar.zst", shell=True, check=True)
subprocess.run("tar --zstd -xf /tmp/ollama.tar.zst -C /usr && rm -f /tmp/ollama.tar.zst", shell=True, check=True)

print("2/4 Starting Ollama daemon with performance settings...")
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_KEEP_ALIVE"] = "24h"          # Keep models loaded to stop VRAM reload delay timeouts
env["OLLAMA_FLASH_ATTENTION"] = "1"        # Accelerate prompt prefill calculation
env["OLLAMA_MAX_LOADED_MODELS"] = "1"      # Force 1 model in VRAM to prevent GPU Out-Of-Memory
env["OLLAMA_NUM_PARALLEL"] = "1"
env["OLLAMA_GPU_OVERHEAD"] = "0"
subprocess.Popen(["ollama", "serve"], env=env)

# Wait for daemon
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/", timeout=1)
        print("✓ Ollama daemon active on port 11434")
        break
    except Exception:
        time.sleep(1)

# List of new benchmark models to mount
MODELS_TO_PULL = [
    "mistral-nemo:12b",
    "qwen2.5:14b",
    "codellama:7b",
    "llama3.2:3b",
]

print("3/4 Pulling new model suite...")
for model in MODELS_TO_PULL:
    print(f"--> Pulling {model}...")
    subprocess.run(["ollama", "pull", model], check=True)

print("4/4 Starting Cloudflare Tunnel...")
subprocess.run("curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared", shell=True, check=True)

proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
tunnel_url = None
for line in proc.stdout:
    match = url_pattern.search(line)
    if match:
        tunnel_url = match.group(0)
        print("\n" + "="*65)
        print(f"🎉 COLAB MULTI-MODEL NODE IS LIVE!")
        print(f"Tunnel URL:  {tunnel_url}")
        print(f"Mounted Models: {', '.join(MODELS_TO_PULL)}")
        print("="*65 + "\n")
        break

# Keep session running
while True:
    time.sleep(260000)